# Capítulo 4: Selección ARIMA — Criterios BIC y HQIC

## 4.1 Comparación de criterios de información

Los tres criterios penalizan la complejidad del modelo de forma diferente:

| Criterio | Penalización | Expresión |
|----------|-------------|----------|
| **AIC**  | $2k$ | $-2\ln(\hat{L}) + 2k$ |
| **BIC**  | $k\ln(n)$ | $-2\ln(\hat{L}) + k\ln(n)$ |
| **HQIC** | $2k\ln(\ln(n))$ | $-2\ln(\hat{L}) + 2k\ln(\ln(n))$ |

Para $n \approx 3800$ observaciones (BTC completo):
$$\ln(3800) \approx 8.24 \gg 2$$

Por lo tanto BIC penaliza **~4 veces más** que AIC por parámetro adicional, favoreciendo modelos de menor orden.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import pmdarima as pm
import yfinance as yf
from datetime import datetime

raw = yf.download('BTC-USD', start='2014-09-17', end=datetime.today().strftime('%Y-%m-%d'), progress=False)
btc = raw[['Close']].copy()
btc.columns = ['Price']
btc.index = pd.DatetimeIndex(btc.index).normalize()
btc.dropna(inplace=True)

HORIZONTES = [7, 14, 21, 28]
N = len(btc)
splits = {h: {'train': btc.Price.iloc[:N-h], 'test': btc.Price.iloc[N-h:]} for h in HORIZONTES}

def seleccionar_arima(train, criterion='aic'):
    return pm.auto_arima(train, d=1, start_p=0, max_p=5, start_q=0, max_q=5,
                         information_criterion=criterion, stepwise=True,
                         seasonal=False, suppress_warnings=True, error_action='ignore')

modelos_bic  = {}
modelos_hqic = {}

print("  Horizonte   AIC (p,d,q)    BIC (p,d,q)   HQIC (p,d,q)")
print("-" * 60)

# Recomputar AIC para comparacion
modelos_aic_ref = {h: seleccionar_arima(splits[h]['train'], 'aic') for h in HORIZONTES}

for h in HORIZONTES:
    mb = seleccionar_arima(splits[h]['train'], 'bic')
    mh = seleccionar_arima(splits[h]['train'], 'hqic')
    modelos_bic[h]  = {'model': mb, 'order': mb.order}
    modelos_hqic[h] = {'model': mh, 'order': mh.order}
    print(f"  {h:>6}d    {str(modelos_aic_ref[h].order):>12}  "
          f"{str(mb.order):>12}  {str(mh.order):>12}")

print("-" * 60)

:::{admonition} Conclusión — Criterios BIC y HQIC
:class: tip

BIC y HQIC tienden a seleccionar modelos de igual o menor orden que AIC. Para series largas como BTC ($n > 3000$), la penalización $k\ln(n)$ del BIC es considerablemente mayor que $2k$ del AIC, resultando en modelos más parsimoniosos. La diferencia en performance predictiva entre criterios es marginal para modelos de orden bajo, como se verifica en las métricas de error del Capítulo 8.
:::